# Warlords Multi-Agent Toernooi
Dit notebook draait de Warlords-omgeving vanuit de Arcade Learning Environment (ALE) met 4 custom agenten.

Dit notebook is ontworpen om te draaien in Google Colab. Als je hem lokaal draait, moet je mogelijk meer libraries installeren en controleren of hun versies compatibel zijn.

## 1 Installeer libraries

Voer eerst de onderstaande codecell uit om de libraries te installeren.

In [5]:
# Install the necessary libraries
!pip install pettingzoo[atari]
!pip install "autorom[accept-rom-license]"
!pip install --find-links dist/ --no-cache-dir AutoROM[accept-rom-license]

zsh:1: no matches found: pettingzoo[atari]
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached charset_normalizer-3.4.7-cp312-cp312-macosx_10_13_universal2.whl.metadata (40 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
Using cached click-8.4.1-py3-none-any.whl (116 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached certifi-2026.6.17-py3-none-any.whl (133 kB)
Using cached charset_normalizer-3.4.7-cp312-cp312-macosx_10_13_universal2.whl (311 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
  Created wheel for AutoROM.accept-rom-license: filenam

**Herstart nu je kernel**. Na het herstarten kun je direct doorgaan met de volgende codecel.

## 2 Importeer libraries en download Atari ROMs

Je krijgt een prompt om de AutoROM-overeenkomst te accepteren. Druk op "Y" wanneer je dit ziet.

In [6]:
# Start AutoROM

!AutoROM -y

# Import libraries
from pettingzoo.atari import warlords_v3
from pettingzoo.utils import BaseParallelWrapper
import gymnasium as gym
import numpy as np
from collections import defaultdict, Counter
import importlib
import os
import imageio

/Users/mattias/Documents/GitHub/AutonSys-P3/.venv/lib/python3.12/site-packages/AutoROM/AutoROM.py:264: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/Users/mattias/Documents/GitHub/AutonSys-P3/.venv/lib/python3.12/site-packages/AutoROM/roms
	/Users/mattias/Documents/GitHub/AutonSys-P3/.venv/lib/python3.12/site-packages/multi_agent_ale_py/roms

Existing ROMs will be overwritten.
Installed /Users/mattias/Documents/GitHub/AutonSys-P3/.venv/lib/python3.12/site-packages/AutoROM/roms/adventure.bin
Installed /Users/mattias/Documents/GitHub/AutonSys-P3/.venv/lib/python3.12/site-packages/multi_agent_ale_py/roms/adventure.bin
Installed /Users/mattias/Documents/GitHub/AutonSys-P3/.venv/lib/python3.12/site-packag

## 3 Initialiseer agenten

In deze codecel importeren we de AI-agenten om Warlords te spelen. De bestanden met de agentklassen moeten zich in dezelfde map bevinden als dit notebook.

In [28]:
# Import the agent classes
from random_baseline import RandomAgent
from heuristic_agent import HeuristicAgent
from ppo_agent import PPOAgent
# from agent4 import Agent4

# Instantiate each agent (pass args if needed)
agent_instances = [
    RandomAgent(),
    HeuristicAgent(),
    PPOAgent('./models/ppo_warlords_shared_policy'),
    RandomAgent()
]

agent_names = ['RandomAgent', 'HeuristicAgent', 'PPOAgent', 'RandomAgent2']
scores = defaultdict(int)
wins = Counter()

# Optioneel: zet deze regel aan om RAM-veranderingen tijdens het spel te printen
# agent_instances = wrap_agents_for_ram_debug(agent_instances, agent_names, print_every=20)


## 4 Speel het spel

## 4a RAM-modus

Deze versie gebruikt **RAM-observaties** in plaats van beeldobservaties.

Bij Atari in PettingZoo geeft `obs_type="ram"` een vector van **128 bytes** terug
(= 1024 bits RAM van de Atari-console). Elke entry in de array is dus één RAM-byte.
De posities van spelers, projectielen en andere game-state zitten **gecodeerd in deze bytes**.

Omdat de exacte byte-indexen voor Warlords game-specifiek zijn, zitten hieronder ook debug-tools
om te zien welke RAM-bytes veranderen tijdens het spel. Daarmee kun je gericht achterhalen welke
bytes de paddleposities, balpositie, etc. voorstellen.

In [29]:
# Hulpfuncties voor RAM-observaties
import numpy as np

def describe_ram_observation(observation):
    observation = np.asarray(observation, dtype=np.uint8)
    print("RAM shape:", observation.shape)
    print("dtype:", observation.dtype)
    print("Eerste 32 bytes:", observation[:32].tolist())

def diff_ram(previous_obs, current_obs):
    previous_obs = np.asarray(previous_obs, dtype=np.uint8)
    current_obs = np.asarray(current_obs, dtype=np.uint8)
    changed = np.where(previous_obs != current_obs)[0]
    return [(int(i), int(previous_obs[i]), int(current_obs[i])) for i in changed]

class RAMDebugAgent:
    """
    Wrapper-agent die elke N stappen RAM-veranderingen print.
    Handig om te reverse-engineeren welke bytes overeenkomen met posities.
    """
    def __init__(self, base_agent, name="Agent", print_every=20):
        self.base_agent = base_agent
        self.name = name
        self.print_every = print_every
        self.prev_obs = None
        self.step_idx = 0

    def act(self, observation):
        observation = np.asarray(observation, dtype=np.uint8)

        if self.step_idx == 0:
            print(f"[{self.name}] start-RAM:")
            describe_ram_observation(observation)
        elif self.step_idx % self.print_every == 0:
            changes = diff_ram(self.prev_obs, observation)
            print(f"[{self.name}] stap {self.step_idx}: {len(changes)} bytes veranderd")
            print("Veranderde bytes (index, oud, nieuw):", changes[:20])

        self.prev_obs = observation.copy()
        self.step_idx += 1
        return self.base_agent.act(observation)

def wrap_agents_for_ram_debug(agent_instances, agent_names, print_every=20):
    return [
        RAMDebugAgent(agent, name=name, print_every=print_every)
        for agent, name in zip(agent_instances, agent_names)
    ]

In deze sectie spelen de vier agenten Warlords. Aan het einde van elk spel wordt de score bijgehouden. De winnaar is de agent met de hoogste score.

In [30]:
# Create environment
env = warlords_v3.env(obs_type="ram", render_mode="rgb_array")

# Prepare directory for videos
video_dir = "./warlords_videos"
os.makedirs(video_dir, exist_ok=True)

print("Observatiemodus:", "RAM (128 bytes per observatie)")


Observatiemodus: RAM (128 bytes per observatie)


De volgende codecel speelt het spel.

In [31]:
# Function to run one game and record video
def run_game(game_id):
    env.reset()

    # Map environment agents to their corresponding agent instances
    agent_mapping = {
        env.agents[i]: agent_instances[i]
        for i in range(len(env.agents))
    }

    # Map environment agents to their corresponding agent names
    agent_name_mapping = {
        env.agents[i]: agent_names[i]
        for i in range(len(env.agents))
    }

    # Reset scores
    for agent in agent_names:
        scores[agent] = 0

    done = False
    terminated = False
    truncated = False

    frames = []
    first_observation_logged = False

    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()
        scores[agent_name_mapping[agent]] += reward

        if reward > 0:
            print(f"Agent {agent_name_mapping[agent]} won the game!")
            wins[agent_name_mapping[agent]] += 1

        # Use this to debug
        # print(f"Agent: {agent}, Name: {agent_name_mapping[agent]}, Reward: {reward}, Termination: {termination}, Truncation: {truncation}")

        # Deze check hoort binnen de for-loop te vallen
        if not first_observation_logged:
            obs_arr = np.asarray(observation)
            print(f"Eerste observatie voor {agent_name_mapping[agent]}: shape={obs_arr.shape}, dtype={obs_arr.dtype}")
            print("Eerste 32 RAM-bytes:", obs_arr[:32].tolist())
            first_observation_logged = True

        if termination or truncation:
            action = None
        else:
            # this is where you would insert your policy
            agent_obj = agent_mapping[agent]
            action = agent_obj.act(observation)

        env.step(action)

        # Capture the rendered frame
        frame = env.render()
        frames.append(frame)

    env.close()

    # Save video using imageio
    video_path = os.path.join(video_dir, f"game_{game_id}.mp4")
    imageio.mimsave(video_path, frames, fps=15)

In [32]:
# Run 10 games
for game in range(10):
    print(f"Running game {game + 1}...")
    run_game(game_id=game)

print("\nFinal Scores:")
for agent in agent_names:
    print(f"{agent}: Total Reward = {scores[agent]}, Wins = {wins[agent]}")

try:
    winner = wins.most_common(1)[0]
    print(f"Winner: {winner[0]} with {winner[1]} wins!")
except IndexError:
    print("No winners found.")

Running game 1...
Eerste observatie voor RandomAgent: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Agent RandomAgent2 won the game!
Running game 2...
Eerste observatie voor RandomAgent: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Agent RandomAgent won the game!
Running game 3...
Eerste observatie voor RandomAgent: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


...

Agent RandomAgent won the game!
Running game 9...
Eerste observatie voor RandomAgent: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Agent RandomAgent2 won the game!
Running game 10...
Eerste observatie voor RandomAgent: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Agent RandomAgent won the game!

Final Scores:
RandomAgent: Total Reward = 1, Wins = 3
HeuristicAgent: Total Reward = -1, Wins = 0
PPOAgent: Total Reward = -1, Wins = 0
RandomAgent2: Total Reward = -1, Wins = 7
Winner: RandomAgent2 with 7 wins!


In [27]:
# Display download links for videos
import glob
from IPython.display import FileLink, display
video_files = sorted(glob.glob(f"{video_dir}/*.mp4"))
print("\nDownload the recorded games:")
for file in video_files:
    display(FileLink(file))


Download the recorded games:


/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/evaluation_episode_1.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_0.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_1.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_2.mp4

...

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_5.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_6.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_7.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_8.mp4

/Users/mattias/Documents/GitHub/AutonSys-P3/warlords_videos/game_9.mp4